# Segment delay breakdown (MIT thesis §3.3)

(this was an attempt to one-shot the intersection delay analysis - it really didn't work, but was usefrul for figuring out some initial challenges - e.g. stops recorded just after signal bars are hard, bus speeds relatively rarely go to 0, may need to use some more probabalistic modeling as suggested in the paper to calculate)

For each signal-bounded segment and each daytime trip, decompose observed travel time (Eq 3.6):

`T_obs = T_ff + T_dwell + D_signal + D_congestion + Loss`

- **Segments** run between consecutive signals; segments that END at a near-side signal are ignored.
- **T_ff** (free-flow) = 5th-percentile travel time of nighttime trips (start 21:00–06:00).
- **T_dwell**: stops near a bus stop or near-side signal are treated as dwell and excluded from signal/congestion.
- **D_signal = uniform + overflow**, measured from stop durations: the stop closest to the downstream signal (within 100 m) is uniform, capped at the red-phase length (95th-pct of night stop-bar stops); additional upstream stops within 200 m connected by creeping (stop-slow-stop, no faster period) are overflow.
- **D_congestion**: residual (Eq 3.21) = max(0, T_obs − T_ff − T_dwell − D_signal). Loss is neglected.

All calculations live in `segment_delay.py`; tunable parameters (night window, percentiles, signal-stop area, faster-period speed) are module constants there.

In [1]:
import geopandas as gpd
import pandas as pd
from IPython.display import display

from constants import CA_NAD83_Albers, CULVER_CITY_FEED_KEY, SERVICE_DATE, SHAPE_KEY_TO_SHAPE_ID_MAP
from _data_loaders import (
    get_culver_city_vehicle_positions,
    get_selected_shapes,
    get_traffic_signals,
    list_available_service_dates,
)
from segment_delay import run_segment_delay_analysis

## Configuration

In [2]:
SHAPE_KEY = "105"
SHAPE_ID = SHAPE_KEY_TO_SHAPE_ID_MAP[SHAPE_KEY]
print(f"Shape: {SHAPE_KEY} -> {SHAPE_ID}")

Shape: 105 -> shp-1-05


## Load inputs

All trips for the shape across every service date, plus the shape geometry, signals, and stops (with hand-curated near-side flags). Loading all dates reads every per-date geoparquet, so this cell is the slow one.

In [3]:
service_dates = list_available_service_dates()
vehicle_positions = pd.concat(
    [
        get_culver_city_vehicle_positions([SHAPE_KEY], service_date).assign(service_date=service_date)
        for service_date in service_dates
    ],
    ignore_index=True,
)
print(f"{len(vehicle_positions):,} positions over {len(service_dates)} dates")

shapes = get_selected_shapes(SERVICE_DATE, CULVER_CITY_FEED_KEY, [SHAPE_ID])
signals = get_traffic_signals()
stops = gpd.read_file(f"data/stops_{SHAPE_ID}.geojson").to_crs(CA_NAD83_Albers)

293,339 positions over 22 dates


## Run the breakdown

In [4]:
result = run_segment_delay_analysis(vehicle_positions, shapes, signals, stops, SHAPE_ID)
print(f"{len(result.segments)} analyzed segments (between signals, not ending at a near-side signal)")
result.segments.round(1)

44 analyzed segments (between signals, not ending at a near-side signal)


,start_signal,end_signal,start_distance_m,end_distance_m,length_m
segment_id,,,,,
sig1_to_sig2,1,2,27.5,386.5,359.0
sig2_to_sig3,2,3,386.5,517.1,130.5
sig3_to_sig4,3,4,517.1,981.9,464.8
sig4_to_sig5,4,5,981.9,1030.1,48.2
sig5_to_sig6,5,6,1030.1,1236.4,206.4
sig6_to_sig7,6,7,1236.4,1494.1,257.7
sig7_to_sig8,7,8,1494.1,1694.7,200.6
sig8_to_sig9,8,9,1694.7,1803.6,108.9
sig9_to_sig10,9,10,1803.6,2010.9,207.3


## Trip numbers per segment

Daytime trips per segment: total, how many stopped at the signal, how many had overflow, how many had any congestion delay, plus the nighttime sample size behind each segment's baselines.

In [5]:
result.trip_counts

,n_day_trips,n_signal_stop,n_overflow,n_congestion,n_night_trips
segment_id,,,,,
sig1_to_sig2,549,0,0,517,45
sig2_to_sig3,670,13,9,656,57
sig3_to_sig4,671,0,0,645,57
sig4_to_sig5,671,5,0,651,57
sig5_to_sig6,671,1,0,650,57
sig6_to_sig7,672,0,0,658,57
sig7_to_sig8,673,1,0,640,57
sig8_to_sig9,673,19,0,635,57
sig9_to_sig10,673,14,0,643,57


## Raw travel times per segment

Descriptive statistics of observed daytime segment travel times (seconds), with the free-flow travel time (T_ff) alongside. The matrix below is one daytime trip per row, one segment per column.

In [6]:
display(result.raw_travel_times.round(1))
display(result.travel_time_matrix.round(1))

,count,mean,std,min,25%,50%,75%,max,free_flow_travel_time_s
segment_id,,,,,,,,,
sig10_to_sig11,675.0,45.1,17.8,18.2,32.1,41.6,53.6,133.8,26.9
sig11_to_sig12,675.0,54.5,28.3,18.4,35.9,48.0,66.5,360.8,26.4
sig12_to_sig13,675.0,40.0,25.1,10.9,21.6,32.9,51.0,167.5,13.7
sig13_to_sig14,676.0,43.5,23.9,12.9,25.8,37.4,52.7,143.5,20.8
sig14_to_sig15,679.0,46.0,25.7,10.6,28.0,38.5,58.7,235.9,19.4
sig15_to_sig16,680.0,32.6,22.9,6.1,17.9,27.6,39.5,199.4,13.1
sig16_to_sig17,683.0,19.0,13.2,3.4,11.0,15.5,23.5,88.3,6.8
sig1_to_sig2,549.0,50.8,23.8,19.8,37.7,46.5,58.8,412.9,23.8
sig20_to_sig21,694.0,41.9,50.5,12.6,20.0,25.8,37.7,511.9,14.8


segment_id             sig10_to_sig11  sig11_to_sig12  sig12_to_sig13  \
service_date TRIP_KEY                                                   
2026-01-31   1016.0              49.4            29.5            39.2   
             1053.0              24.9            51.8            38.8   
             1068.0              37.8            41.9            23.4   
             1089.0              31.6            41.8            98.6   
             1156.0              81.9            72.9            60.0   
...                               ...             ...             ...   
2026-02-16   858.0               27.7            41.7            30.6   
             876.0               28.5            34.1            38.5   
             943.0               61.7            74.4            29.4   
             972.0               37.8            30.3            18.2   
             986.0               56.7            53.3            87.9   

segment_id             sig13_to_sig14  sig14_to_sig15  sig15_to_sig16  \
service_date TRIP_KEY                                                   
2026-01-31   1016.0              31.1            38.3            20.9   
             1053.0              87.3            71.6            23.4   
             1068.0              23.7            33.0            21.0   
             1089.0              42.8            31.4            13.1   
             1156.0              54.2            45.0            43.1   
...                               ...             ...             ...   
2026-02-16   858.0              138.5            16.9            18.7   
             876.0              103.9            45.5            14.8   
             943.0               21.2            24.3            52.2   
             972.0               61.6            40.5            13.2   
             986.0               86.0            51.4            39.3   

segment_id             sig16_to_sig17  sig1_to_sig2  sig20_to_sig21  \
service_date TRIP_KEY                                                 
2026-01-31   1016.0              11.9          37.7            53.6   
             1053.0              10.4          49.4            22.2   
             1068.0               8.7          38.8            19.7   
             1089.0               5.0          79.7           253.8   
             1156.0              44.2          48.7            30.6   
...                               ...           ...             ...   
2026-02-16   858.0               87.5          55.9            20.4   
             876.0                7.9          27.2            22.3   
             943.0               22.7          43.7            22.9   
             972.0                7.7          48.9            35.7   
             986.0               29.8          90.1            18.7   

segment_id             sig22_to_sig23  ...  sig49_to_sig50  sig4_to_sig5  \
service_date TRIP_KEY                  ...                                 
2026-01-31   1016.0              16.6  ...            39.8           3.0   
             1053.0             100.0  ...            43.5           4.9   
             1068.0              76.9  ...            67.2           6.0   
             1089.0              25.1  ...            31.9           3.6   
             1156.0             119.6  ...            41.9           5.8   
...                               ...  ...             ...           ...   
2026-02-16   858.0               69.4  ...            17.7           5.6   
             876.0               19.8  ...            18.9           3.5   
             943.0               89.2  ...            27.9           3.5   
             972.0               41.1  ...            19.3           5.1   
             986.0               11.0  ...            22.9           4.7   

segment_id             sig51_to_sig52  sig52_to_sig53  sig54_to_sig37  \
service_date TRIP_KEY                                                   
2026-01-31   1016.0              41.2    

## Delays per segment

Descriptive statistics (seconds) of each delay component over daytime trips: uniform, overflow, total signal, and congestion delay.

In [7]:
for component in ["uniform_delay_s", "overflow_delay_s", "signal_delay_s", "congestion_delay_s"]:
    print(f"{component} by segment:")
    display(result.component_stats[component].round(1))

uniform_delay_s by segment:


,count,mean,std,min,25%,50%,75%,max
segment_id,,,,,,,,
sig10_to_sig11,675.0,0.0,0.2,0.0,0.0,0.0,0.0,6.0
sig11_to_sig12,675.0,0.0,1.1,0.0,0.0,0.0,0.0,28.0
sig12_to_sig13,675.0,4.3,9.7,0.0,0.0,0.0,0.0,32.6
sig13_to_sig14,676.0,1.1,7.3,0.0,0.0,0.0,0.0,109.0
sig14_to_sig15,679.0,0.9,4.8,0.0,0.0,0.0,0.0,39.8
sig15_to_sig16,680.0,2.4,8.0,0.0,0.0,0.0,0.0,35.3
sig16_to_sig17,683.0,4.1,13.5,0.0,0.0,0.0,0.0,58.6
sig1_to_sig2,549.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
sig20_to_sig21,694.0,0.5,9.4,0.0,0.0,0.0,0.0,201.0


overflow_delay_s by segment:


,count,mean,std,min,25%,50%,75%,max
segment_id,,,,,,,,
sig10_to_sig11,675.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
sig11_to_sig12,675.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
sig12_to_sig13,675.0,2.2,11.9,0.0,0.0,0.0,0.0,167.4
sig13_to_sig14,676.0,0.1,1.8,0.0,0.0,0.0,0.0,32.0
sig14_to_sig15,679.0,0.1,1.3,0.0,0.0,0.0,0.0,21.2
sig15_to_sig16,680.0,1.4,10.7,0.0,0.0,0.0,0.0,168.7
sig16_to_sig17,683.0,0.6,6.0,0.0,0.0,0.0,0.0,108.4
sig1_to_sig2,549.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
sig20_to_sig21,694.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


signal_delay_s by segment:


,count,mean,std,min,25%,50%,75%,max
segment_id,,,,,,,,
sig10_to_sig11,675.0,0.0,0.2,0.0,0.0,0.0,0.0,6.0
sig11_to_sig12,675.0,0.0,1.1,0.0,0.0,0.0,0.0,28.0
sig12_to_sig13,675.0,6.5,18.7,0.0,0.0,0.0,0.0,200.0
sig13_to_sig14,676.0,1.2,7.7,0.0,0.0,0.0,0.0,109.0
sig14_to_sig15,679.0,1.0,5.5,0.0,0.0,0.0,0.0,61.0
sig15_to_sig16,680.0,3.7,16.4,0.0,0.0,0.0,0.0,204.0
sig16_to_sig17,683.0,4.7,16.8,0.0,0.0,0.0,0.0,167.0
sig1_to_sig2,549.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
sig20_to_sig21,694.0,0.5,9.4,0.0,0.0,0.0,0.0,201.0


congestion_delay_s by segment:


,count,mean,std,min,25%,50%,75%,max
segment_id,,,,,,,,
sig10_to_sig11,675.0,18.1,17.1,0.0,5.1,14.2,26.3,106.9
sig11_to_sig12,675.0,26.2,22.3,0.0,9.5,21.4,38.7,182.9
sig12_to_sig13,675.0,19.3,14.3,0.0,7.2,18.1,28.5,74.9
sig13_to_sig14,676.0,21.8,21.2,0.0,5.0,16.5,31.4,122.6
sig14_to_sig15,679.0,23.1,19.3,0.0,8.4,18.2,34.6,133.8
sig15_to_sig16,680.0,16.4,15.8,0.0,4.6,14.1,23.8,169.5
sig16_to_sig17,683.0,9.1,8.3,0.0,2.6,7.3,13.8,56.9
sig1_to_sig2,549.0,24.9,23.5,0.0,12.6,21.6,32.6,389.1
sig20_to_sig21,694.0,22.3,40.6,0.0,5.0,10.9,21.5,497.1


## Segment summary

One row per segment: geometry, free-flow time, red-phase cap, trip counts, and mean uniform / overflow / signal / congestion / dwell.

In [8]:
result.segment_summary.round(1)

,start_signal,end_signal,start_distance_m,end_distance_m,length_m,free_flow_travel_time_s,red_phase_cap_s,n_day_trips,n_signal_stop,n_overflow,n_congestion,n_night_trips,mean_uniform_delay_s,mean_overflow_delay_s,mean_signal_delay_s,mean_congestion_delay_s,mean_total_delay_s,mean_dwell_s
segment_id,,,,,,,,,,,,,,,,,,
sig1_to_sig2,1,2,27.5,386.5,359.0,23.8,NaN,549,0,0,517,45,0.0,0.0,0.0,24.9,24.9,3.9
sig2_to_sig3,2,3,386.5,517.1,130.5,7.8,16.0,670,13,9,656,57,0.3,0.2,0.5,14.3,14.9,0.9
sig3_to_sig4,3,4,517.1,981.9,464.8,32.5,NaN,671,0,0,645,57,0.0,0.0,0.0,28.4,28.4,0.5
sig4_to_sig5,4,5,981.9,1030.1,48.2,3.0,NaN,671,5,0,651,57,0.2,0.0,0.2,3.3,3.4,0.0
sig5_to_sig6,5,6,1030.1,1236.4,206.4,12.7,NaN,671,1,0,650,57,0.2,0.0,0.2,12.4,12.6,0.8
sig6_to_sig7,6,7,1236.4,1494.1,257.7,16.8,NaN,672,0,0,658,57,0.0,0.0,0.0,13.9,13.9,0.1
sig7_to_sig8,7,8,1494.1,1694.7,200.6,15.0,NaN,673,1,0,640,57,0.1,0.0,0.1,9.1,9.2,0.0
sig8_to_sig9,8,9,1694.7,1803.6,108.9,8.4,NaN,673,19,0,635,57,0.5,0.0,0.5,8.9,9.4,0.3
sig9_to_sig10,9,10,1803.6,2010.9,207.3,30.4,67.8,673,14,0,643,57,0.5,0.0,0.5,28.2,28.6,7.2


## Per trip-segment detail

Every daytime (trip, segment) with observed travel time, free-flow time, dwell, and the uniform / overflow / signal / congestion / total delay components.

In [9]:
result.daytime_delays.round(1)

,service_date,TRIP_KEY,period,segment_id,observed_travel_time_s,dwell_s,uniform_stop_duration_s,overflow_stop_duration_s,has_signal_stop,free_flow_travel_time_s,red_phase_cap_s,uniform_delay_s,overflow_delay_s,signal_delay_s,congestion_delay_s,total_delay_s
0,2026-01-31,1016.0,day,sig1_to_sig2,37.7,0.0,0.0,0.0,False,23.8,NaN,0.0,0.0,0.0,13.9,13.9
1,2026-01-31,1016.0,day,sig2_to_sig3,11.2,0.0,0.0,0.0,False,7.8,16.0,0.0,0.0,0.0,3.3,3.3
2,2026-01-31,1016.0,day,sig3_to_sig4,37.3,0.0,0.0,0.0,False,32.5,NaN,0.0,0.0,0.0,4.8,4.8
3,2026-01-31,1016.0,day,sig4_to_sig5,3.0,0.0,0.0,0.0,False,3.0,NaN,0.0,0.0,0.0,0.0,0.0
4,2026-01-31,1016.0,day,sig5_to_sig6,12.4,0.0,0.0,0.0,False,12.7,NaN,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32761,2026-02-16,986.0,day,sig47_to_sig48,39.9,0.0,0.0,0.0,False,21.9,NaN,0.0,0.0,0.0,18.0,18.0
32762,2026-02-16,986.0,day,sig48_to_sig49,54.5,0.0,0.0,0.0,False,37.0,NaN,0.0,0.0,0.0,17.5,17.5
32763,2026-02-16,986.0,day,sig49_to_sig50,22.9,0.0,0.0,0.0,False,16.2,NaN,0.0,0.0,0.0,6.7,6.7
32764,2026-02-16,986.0,day,sig51_to_sig52,21.7,0.0,0.0,0.0,False,18.3,NaN,0.0,0.0,0.0,3.5,3.5


## Map of segments

Each analyzed segment is drawn along the shape in a distinct color so adjacent segments are easy to tell apart. Click a segment to see all of its summary values (free-flow time, red-phase cap, trip counts, mean delay components).

In [10]:
import folium
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt

from segment_delay import segment_geometries

segment_geometry = segment_geometries(shapes, SHAPE_ID, result.segments)[["geometry"]]
segment_map = gpd.GeoDataFrame(
    result.segment_summary.round(1).join(segment_geometry),
    geometry="geometry",
    crs=shapes.crs,
).reset_index().to_crs("EPSG:4326")

# Give each segment a distinct color (cycled in route order) so neighbors differ.
palette = [mcolors.to_hex(c) for c in plt.get_cmap("tab20").colors]
segment_map["color"] = [palette[i % len(palette)] for i in range(len(segment_map))]

popup_fields = [c for c in segment_map.columns if c not in ("geometry", "color")]

min_lon, min_lat, max_lon, max_lat = segment_map.total_bounds
segment_delay_map = folium.Map(
    location=[(min_lat + max_lat) / 2, (min_lon + max_lon) / 2],
    zoom_start=13,
    tiles="CartoDB positron",
)
folium.GeoJson(
    segment_map,
    style_function=lambda feature: {
        "color": feature["properties"]["color"],
        "weight": 5,
        "opacity": 0.85,
    },
    popup=folium.GeoJsonPopup(fields=popup_fields),
).add_to(segment_delay_map)
segment_delay_map

In [11]:
example_detail = analyze_trip_segment(
    vehicle_positions, shapes, signals, stops, SHAPE_ID,
    "2026-02-13", 1376, "sig40_to_sig41"
)
plot_trip_segment(
    example_detail,
    title=f"{example.example}: {example.segment_id} — {example.service_date} trip {example.TRIP_KEY}",
)
plt.show()

NameError: name 'analyze_trip_segment' is not defined